# 第8回 データサイエンス演習：データの可視化（グラフ作成）

## 本講義の目標
数値の羅列であるデータから，特徴や傾向を視覚的に読み取る「データ可視化」の手法を学ぶ．
本講義では，「まずはPandasで素早くグラフの形にし，必要な時にMatplotlibを導入して細かい仕上げを行う」という，データ分析の実務に即したワークフローを習得する．

---

## 1. 準備：データの読み込み

まずは，前回の演習で使用したデータ（`floodgate_data2026.csv`）と，データ分析の基本ツールであるPandasを読み込む．
（※事前に左側のフォルダアイコンから，CSVファイルをアップロードしておくこと．
`floodgate_data2026.csv`は[https://github.com/k0-1/data_science_2026/tree/main/sessions/07_pandas](https://github.com/k0-1/data_science_2026/tree/main/sessions/07_pandas)からダウンロードできる．)

In [ ]:
import pandas as pd

# データの読み込み
df = pd.read_csv('floodgate_data2026.csv')
display(df.head())

---

## 2. Pandasの描画機能の全体像

Pandasのデータフレーム（表）には，自分自身をグラフ化する `.plot` という機能が備わっている．
分析の目的に合わせて，以下のような多様なグラフを1行のコードで使い分けることができる．

* **`.plot.line()`** ： 折れ線グラフ（時系列データの推移などを確認する）
* **`.plot.bar()`** ： 棒グラフ（カテゴリごとの大小を比較する）
* **`.plot.hist()`** ： ヒストグラム（データの散らばり具合や分布を確認する）
* **`.plot.scatter()`** ： 散布図（2つの変数の間に関係性があるかを確認する）

これらはSeriesやDataframeに対して使用することができる．
ただし，Dataframeの場合はデータが''そろって''いないとあまり意味のないグラフになる．

他にもいろいろあるので気になった人は調べてみよう．

---

## 3. ヒストグラムとデータ単位の確認（分布を見る）

まずは，将棋AIのレーティングの分布をヒストグラム（度数分布表）で確認してみよう．
対局数の偏りに影響されない「純粋なAIの実力分布」を知るためには，前回の講義で学んだ `groupby` を用いて，AIごとに最高レーティングを1つだけ抽出（重複を排除）して描画する．

In [ ]:
# 先手と後手のデータを縦に繋ぎ、AIごとに「最高レーティング」を抽出する
df_sente = df[['先手', '先手レーティング']].rename(columns={'先手': 'AI名', '先手レーティング': 'レーティング'})
df_gote = df[['後手', '後手レーティング']].rename(columns={'後手': 'AI名', '後手レーティング': 'レーティング'})
df_all = pd.concat([df_sente, df_gote])

# AIごとにデータを集約（重複の排除）
ai_unique_ratings = df_all.groupby('AI名')['レーティング'].max()

# 行の個数を見ることでどれぐらいのAIが参加しているかを表示する．
print(len(ai_unique_ratings))

# 重複をなくしたデータで、ヒストグラムを描く
# `bins`には棒の数を書く．多いほど，細かい棒になる．
ai_unique_ratings.plot.hist(bins=30)

---

## 4. 棒グラフによる比較と可視化（大小を比べる）

次に，先ほど集計したデータを用いて「どのAIのレーティングが高いか」を棒グラフで比較してみる．数百のAIをすべて棒グラフにすると横軸が潰れてしまうため，`.head(10)` を用いて上位のみを抽出して比較する．
棒グラフを表示するには`plot.bar()`を用いる．


In [ ]:
# ランキングトップ10の抽出と、棒グラフの描画
top10_ai = ai_unique_ratings.sort_values(ascending=False).head(10)
top10_ai.plot.bar()

**※注意：画面に出力されたグラフを見ると，横軸などの日本語'AI名'が「AI□」と文字化けしている．この解消方法は後述する．**


### 練習問題：「手数」の分布はどうなっているか．
将棋の対局が，だいたい何手くらいで終わるのか（手数の散らばり具合）を確認したいとする．現在のデータ（`df`）の「手数」という連続した数値の分布を見るのに適したグラフを描画しよう．

<details>
<summary>解答例を見る（クリックで展開）</summary>

>```python
># 練習1のコードをここに記述すること
>df['手数'].plot.hist(bins=50)
>```
>100～150手あたりに大きな山があることが視覚的にわかる．
>棒の数（bins）は50程度に設定すると見やすくなる）
</details>

### 「勝者」の割合（先手・後手・引き分け）はどうなっているか．
データ全体の中で，「先手」「後手」「引き分け」の割合を比較し可視化したい．
これらは手数のような連続した数値ではなく独立したカテゴリ（種類）である．
そのような場合は棒グラフを用いるとよい．

#### データのカウントと割合の算出（`value_counts`）
まずは`df`の`勝者`から先手，後手，引き分けをカウントしよう
このような時は`value_counts`を用いる．

In [ ]:
df['勝者'].value_counts()

割合を知りたいときは，オプションとして`normalize=True`と書く．

In [ ]:
win_rates=df['勝者'].value_counts(normalize=True)
display(win_rates)

これを棒グラフで可視化したいときは`plot.bar`を用いる．

In [ ]:
win_rates.plot.bar()

文字化けしていて何の棒なのかわからない．
そこでツールを日本語を扱えるようにするツールを導入する．

---

## 5. Matplotlibを用いたグラフの装飾（仕上げ）

Pandasは素早い描画に優れているが，「日本語の文字化け（□□□）」や「タイトルがない」など，他者に見せる資料としては不十分である．ここで，グラフ装飾の専用ライブラリ **Matplotlib** と，日本語化ツール（`japanize-matplotlib`）を導入する．

In [ ]:
# 日本語化ツールのインストール
!pip install japanize-matplotlib

In [ ]:
# ツールの読み込み
import matplotlib.pyplot as plt
import japanize_matplotlib

表示したいグラフ`plot.bar`と同じセル内に
matplotlibのコマンドを書き，
`plt.show()`とすると，装飾されたグラフが得られる．

`plt.savefig(ファイル.拡張子)`と`plt.show()`の前に書くと，
グラフが`ファイル.拡張子`としてGoogle Colabに保存される．

In [ ]:
# 1. Pandasによる描画（グラフの骨組み）
top10_ai.plot.bar()

# 2. Matplotlibによる装飾（仕上げ）
plt.title('最強将棋AI レーティングトップ10', fontsize=16) # タイトルをつける  
plt.xlabel('将棋AIの名称', fontsize=12) # x軸のラベルをつける
plt.ylabel('最高レーティング', fontsize=12) #y軸のラベルをつける
plt.xticks(rotation=45)                                   

#グラフをGoogle Colabに保存
plt.savefig('top10_ai.png', bbox_inches='tight') 

# 3. グラフの表示
plt.show()

`bbox_inches='tight'`は凡例や文字などもピッタリ収めるという意味．

前回と同様にGoogle Colabからローカルにダウンロードするには次のコマンドを実行すればよい．

In [ ]:
from google.colab import files
files.download('top10_ai.png')

### 練習問題
先手後手の勝率を表すグラフに
'先手・後手の勝率'というタイトルをつけ，縦軸に'割合'というラベルをつけて表示させてください．

表示から先手の棒が，後手の棒よりも少しだけ高くなっている（先手有利）ことが一目でわかる．

---

## 6. 散布図（Scatter）による「関係性」の可視化

2つの変数の間にどのような関係性があるかを調べるには，**散布図（`.plot.scatter()`）**を使う．
例として，「先手レーティング」と「後手レーティング」の関係を見てみよう．大量のデータを打つため，`alpha`（透明度）を指定して重なりを見やすくするのが鉄則である．

In [ ]:
# alpha=0.1 で半透明（10%の濃さ）にして散布図を描画
df.plot.scatter(x='先手レーティング', y='後手レーティング', alpha=0.1, figsize=(8, 8), color='blue')

import matplotlib.pyplot as plt
plt.title('先手レーティングと後手レーティングの関係')
plt.show()

右肩上がりの帯ができることから，「実力が近いAI同士がマッチングされている」ことがわかる．

---

## 【提出課題】「実力差」と「手数」の関係性の調査

将棋において，「先手と後手の実力が拮抗している対局ほど，互いに決定打を与えられずに終盤までもつれ込む（手数が長くなる）」という仮説について，散布図を用いて視覚的に調査すること．

### 【課題の指示】
以下の条件を満たすコードを記述・実行し，出力されたグラフを提出すること．

1. **データの前処理**
   `df`を用い，「先手レーティング」から「後手レーティング」を引いた値を計算し，新しい列 `'レーティング差'` を作成すること．（プラスなら先手が格上，マイナスなら後手が格上）
2. **基本描画**
   横軸を `'レーティング差'`，縦軸を `'手数'` とした散布図を作成すること．
3. **グラフの装飾**
   `plt.title()` で'レーティング差と手数の関係'とタイトルを付け，`alpha=0.3` を指定して点を半透明にすること．
4. 得られたグラフを画像ファイルとして保存し，その画像ファイルを提出してください．

## 【発展的な内容】データから多角的に見る

描画した散布図を見ると，中央付近が最も手数が長く（盛り上がって）おり，実力差が開く（左右に行く）ほどあっさりと短手数で決着していることがわかる．つまり，この散布図における**手数のピーク（山の頂点）は，「盤上での実力が完全に拮抗しているポイント**とみることができる．

では，最も手数が長くなるのは，レーティング差がピタリと『0』の時だろうか．
全体の傾向をわかりやすくするため，レーティング差を50刻みにまとめ，平均手数の折れ線グラフを描いてみよう．

In [ ]:
# 1. 散布図の描画
df.plot.scatter(x='レーティング差', y='手数', alpha=0.1, figsize=(10, 6), color='gray')

# 2. レーティング差50刻みの平均手数 
df['レーティング差_50刻み'] = (df['レーティング差'] // 50) * 50
mean_moves = df.groupby('レーティング差_50刻み')['手数'].mean()
plt.plot(mean_moves.index, mean_moves.values, color='red', linewidth=3, label='平均手数')

# 3. グラフの装飾
plt.title('レーティング差と手数の関係', fontsize=16)
plt.xlabel('レーティング差（プラス:先手格上 / マイナス:後手格上）', fontsize=12)
plt.ylabel('手数', fontsize=12)
plt.axvline(x=0, color='blue', linestyle='--', label='レーティング差0')
plt.legend()
plt.show()

### 別のアプローチから見る「先手有利」

赤い折れ線グラフのピークに注目してほしい．青い点線（差が0）の場所ではなく，少しだけ左側（マイナス領域＝後手の方がレーティングが高い状態）にズレている．

勝率の計算でみたように，将棋には 先手有利（勝率が高い）という法則があった．
手数のピークが「後手が少し格上」の場所に存在するということは，**先手には，後手の実力の高さを相殺してしまうほどのシステム上の有利さが最初から存在している**ことを意味している．

勝率という直接的なデータだけでなく，手数の長さという全く別のデータからも「先手有利」が観察できた．このように複数の視点から整合性を確認することが，データ分析の醍醐味である．